# ChemUnited Workflow — API Client

Step-by-step walkthrough of the `custom_project` API.  
Start the server first:
```
python examples\custom_project --port 3116
```

In [11]:
import json
import requests

BASE = "http://127.0.0.1:3116"
session = requests.Session()

def pp(data):
    """Pretty-print JSON."""
    print(json.dumps(data, indent=2, default=str))

## Step 1 — Inspect processes and their schemas

In [13]:
# List all registered processes
resp = session.get(f"{BASE}/processes/")
resp.raise_for_status()
processes = resp.json()
pp(processes)

[
  {
    "name": "clean",
    "description": "User-defined workflow process.",
    "config_schema": {
      "properties": {},
      "title": "ProcessConfig",
      "type": "object"
    }
  },
  {
    "name": "MultiThreading",
    "description": "User-defined workflow process.",
    "config_schema": {
      "properties": {},
      "title": "ProcessConfig",
      "type": "object"
    }
  },
  {
    "name": "React",
    "description": "User-defined workflow process.",
    "config_schema": {
      "properties": {},
      "title": "ProcessConfig",
      "type": "object"
    }
  },
  {
    "name": "SimpleMultithreading",
    "description": "User-defined workflow process.",
    "config_schema": {
      "properties": {},
      "title": "ProcessConfig",
      "type": "object"
    }
  }
]


In [14]:
# Inspect the full parameter schema for one process
process_name = processes[0]["name"]
resp = session.get(f"{BASE}/processes/{process_name}/schema")
resp.raise_for_status()
pp(resp.json())

{
  "process": "clean",
  "config_schema": {
    "properties": {},
    "title": "ProcessConfig",
    "type": "object"
  },
  "main_parameter_schema": {
    "properties": {
      "repetition_ractions": {
        "default": 1,
        "description": "Repetition of the reaction caaried out",
        "editable": true,
        "group": "General",
        "maximum": 100,
        "minimum": 0,
        "title": "Repetition of the reaction",
        "type": "integer",
        "visible": true
      }
    },
    "title": "MainParameter",
    "type": "object"
  }
}


## Step 2 — Browse existing snapshots

In [15]:
# List all snapshots (sorted most-recent first)
resp = session.get(f"{BASE}/snapshots/")
resp.raise_for_status()
snapshots = resp.json()
pp(snapshots)

[
  {
    "filename": "client_test_2026-05-20T11-16-22.json",
    "modified": "2026-05-20T11:16:22.041793",
    "size_bytes": 80
  },
  {
    "filename": "client_test_2026-05-20T10-57-21.json",
    "modified": "2026-05-20T10:57:21.090844",
    "size_bytes": 80
  },
  {
    "filename": "test_2026-05-15T10-38-00.json",
    "modified": "2026-05-15T12:38:00.681447",
    "size_bytes": 116
  }
]


In [16]:
# Read the most recent snapshot in full detail
latest_filename = snapshots[0]["filename"]
resp = session.get(f"{BASE}/snapshots/{latest_filename}")
resp.raise_for_status()
existing_snapshot = resp.json()
print(f"Snapshot: {latest_filename}")
pp(existing_snapshot)

Snapshot: client_test_2026-05-20T11-16-22.json
{
  "main_parameter": {
    "repetition_ractions": 1
  },
  "clean_0": {}
}


## Step 3 — Create a new snapshot

The `data` dict must contain a `main_parameter` key plus one key per process step  
in `{process_name}_{index}` format. Insertion order defines execution sequence.

In [17]:
new_snapshot_body = {
    "name": "client_test",
    "data": {
        "main_parameter": {
            "repetition_ractions": 1
        },
        "clean_0": {}
    }
}

resp = session.post(f"{BASE}/snapshots/", json=new_snapshot_body)
resp.raise_for_status()
created = resp.json()
snapshot_filename = created["filename"]
print(f"Created: {snapshot_filename}")

Created: client_test_2026-05-20T11-30-36.json


## Step 4 — Run the snapshot

`dry_run=True` suppresses all HTTP calls to physical devices.  
Set it to `False` when running against real hardware.

In [18]:
resp = session.post(
    f"{BASE}/run/",
    json={"snapshot": snapshot_filename, "dry_run": True},
)
resp.raise_for_status()
run_id = resp.json()["run_id"]
print(f"Run started — run_id: {run_id}")

Run started — run_id: 17a5a693-478b-43db-ad19-f88d2e7ca84d


## Step 5 — Stream execution events + live device commands

Runs SSE streaming in a background thread while polling `/run/pool` every 500 ms
for live device commands. Both print to this cell until the run ends.

In [19]:
import threading
import time
from datetime import datetime

def _fmt_event(p):
    ts         = datetime.fromtimestamp(p.get("timestamp", 0)).strftime("%H:%M:%S.%f")[:-3]
    event_type = p.get("event_type", "?")
    node_key   = p.get("node_key")
    node_str   = f"{node_key[0]}@{node_key[1]}" if node_key else ""
    state      = p.get("state") or ""
    method     = p.get("method") or ""
    result     = p.get("result")
    active     = p.get("active_predecessor_count")
    completed  = p.get("completed_predecessor_count")
    msg        = p.get("message", "")

    cols = [
        f"{ts}",
        f"{event_type:<22}",
        f"{node_str:<20}" if node_str else " " * 20,
        f"state={state:<10}" if state  else " " * 16,
        f"method={method:<16}" if method else " " * 23,
        f"result={result}"      if result   is not None else "",
        f"pred={completed}/{active}" if active is not None and (active or completed) else "",
        f"| {msg}",
    ]
    return "  ".join(c for c in cols if c)

stream_done = threading.Event()

def _stream():
    with session.get(f"{BASE}/run/{run_id}/stream", stream=True) as resp:
        resp.raise_for_status()
        for raw_line in resp.iter_lines():
            if not raw_line:
                continue
            line = raw_line.decode() if isinstance(raw_line, bytes) else raw_line
            if not line.startswith("data: "):
                continue
            payload = json.loads(line[6:])
            if "state" in payload and "event_type" not in payload:
                sep = "─" * 80
                print(f"{sep} Run ended: {payload['state'].upper()} {sep}")
            else:
                print(f"[EVENT] {_fmt_event(payload)}")
    stream_done.set()

t = threading.Thread(target=_stream, daemon=True)
t.start()

while not stream_done.is_set():
    r = session.get(f"{BASE}/run/pool")
    if r.ok:
        for cmd in r.json():
            print(f"[POOL]  {cmd.get('component')} | {cmd.get('method')} {cmd.get('command')} | params={cmd.get('params')}")
    time.sleep(0.5)

t.join()

[EVENT] 11:30:47.906  EXECUTION_STARTED                                                                        | Execution started for process CustomProcess at start node 'start'.
[EVENT] 11:30:47.907  ITERATION_STARTED       start@0                                                          | Iteration 0 started at node 'start'.
[EVENT] 11:30:47.908  NODE_WAITING            start@0               state=WAITING                              | Iteration root is ready to run.
[EVENT] 11:30:47.908  NODE_RUNNING            start@0               state=RUNNING     method=start             | Running method 'start'.
──────────────────────────────────────────────────────────────────────────────── Run ended: FINISHED ────────────────────────────────────────────────────────────────────────────────


## Step 6 — Read the final execution report

In [20]:
resp = session.get(f"{BASE}/run/{run_id}/report")
resp.raise_for_status()
report = resp.json()
print(f"Run state: {report['state']}")
for i, result in enumerate(report["results"]):
    print(f"\n--- Process step {i} ---")
    pp(result)

Run state: finished

--- Process step 0 ---
{
  "node_state": {
    "start:0": "COMPLETED",
    "end:0": "COMPLETED",
    "conditional_1:0": "COMPLETED",
    "script_1:0": "INACTIVE",
    "loop_1:0": "INACTIVE",
    "command_1:0": "COMPLETED"
  },
  "node_result": {
    "start:0": true,
    "end:0": true,
    "conditional_1:0": true,
    "script_1:0": null,
    "loop_1:0": null,
    "command_1:0": true
  },
  "node_runtime": {
    "start:0": {
      "status_message": "Node completed with result True.",
      "result": true,
      "error": null,
      "started_at": 93380.8142635,
      "finished_at": 93388.8144751,
      "local_data": {}
    },
    "end:0": {
      "status_message": "Node completed with result True.",
      "result": true,
      "error": null,
      "started_at": 93388.8235622,
      "finished_at": 93388.8235753,
      "local_data": {}
    },
    "conditional_1:0": {
      "status_message": "Node completed with result True.",
      "result": true,
      "error": null,
 

## Step 7 — Read logs

In [21]:
# List all log files (sorted most-recent first)
resp = session.get(f"{BASE}/logs/")
resp.raise_for_status()
logs = resp.json()
pp(logs)

[
  {
    "filename": "client_test_2026-05-20T11-30-36_executed_2026-05-20T11-30-47.log",
    "modified": "2026-05-20T11:30:55.918846",
    "size_bytes": 2755
  },
  {
    "filename": "test_2026-05-15T10-38-00_executed_2026-05-20T11-19-44.log",
    "modified": "2026-05-20T11:20:00.741212",
    "size_bytes": 10036
  },
  {
    "filename": "client_test_2026-05-20T11-16-22_executed_2026-05-20T11-19-01.log",
    "modified": "2026-05-20T11:19:09.945808",
    "size_bytes": 2755
  },
  {
    "filename": "client_test_2026-05-20T11-16-22_executed_2026-05-20T11-16-24.log",
    "modified": "2026-05-20T11:16:33.010938",
    "size_bytes": 2755
  },
  {
    "filename": "client_test_2026-05-20T10-57-21_executed_2026-05-20T10-59-19.log",
    "modified": "2026-05-20T10:59:19.497689",
    "size_bytes": 0
  },
  {
    "filename": "client_test_2026-05-20T10-57-21_executed_2026-05-20T10-57-26.log",
    "modified": "2026-05-20T10:57:34.277985",
    "size_bytes": 2755
  },
  {
    "filename": "test_2026-05-1

In [22]:
# Read the most recent log file — last 50 lines
latest_log = logs[0]["filename"]
resp = session.get(f"{BASE}/logs/{latest_log}", params={"tail": 50})
resp.raise_for_status()
print(resp.json()["content"])

2026-05-20T11:30:47 | INFO     | [PROCESS: clean_0] EXECUTION_STARTED | Execution started for process CustomProcess at start node 'start'.
2026-05-20T11:30:47 | INFO     | [PROCESS: clean_0, NODE: start] ITERATION_STARTED | Iteration 0 started at node 'start'.
2026-05-20T11:30:47 | DEBUG    | [PROCESS: clean_0, NODE: start] WAITING | Iteration root is ready to run.
2026-05-20T11:30:47 | INFO     | [PROCESS: clean_0, NODE: start] RUNNING | Running method 'start'.
2026-05-20T11:30:55 | SUCCESS  | [PROCESS: clean_0, NODE: start] COMPLETED | Node completed with result True.
2026-05-20T11:30:55 | DEBUG    | [PROCESS: clean_0, NODE: conditional_1] WAITING | Activated; waiting for other predecessor resolutions.
2026-05-20T11:30:55 | INFO     | [PROCESS: clean_0, NODE: conditional_1] RUNNING | Running method 'conditional_1'.
2026-05-20T11:30:55 | SUCCESS  | [PROCESS: clean_0, NODE: conditional_1] COMPLETED | Node completed with result True.
2026-05-20T11:30:55 | DEBUG    | [PROCESS: clean_0, N